In [1]:
import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
interactions = pd.read_csv(
    "learntwin_student_interactions.csv"
)

questions = pd.read_csv(
    "learntwin_questions.csv"
)

print("Interactions shape:", interactions.shape)
print("Questions shape:", questions.shape)

print("\nInteraction columns:")
print(interactions.columns.tolist())

print("\nQuestion columns:")
print(questions.columns.tolist())

Interactions shape: (10000, 10)
Questions shape: (500, 3)

Interaction columns:
['student_id', 'attempt', 'question_id', 'concept', 'difficulty', 'correct', 'time_gap_days', 'days_since_practice', 'timestamp', 'knowledge_state']

Question columns:
['question_id', 'concept', 'difficulty']


## Understand the datasets


In [3]:
interactions.head()

,student_id,attempt,question_id,concept,difficulty,correct,time_gap_days,days_since_practice,timestamp,knowledge_state
0,S001,1,Q043,RIGHT JOIN,0.724,0,3.007,3.007,3.007,0.3574
1,S001,2,Q452,RIGHT JOIN,0.577,0,1.471,1.471,4.478,0.3103
2,S001,3,Q022,UNION,0.499,1,1.675,6.154,6.154,0.3849
3,S001,4,Q007,WHERE,0.380,1,0.563,6.717,6.717,0.4015
4,S001,5,Q114,STORED PROCEDURES,0.598,1,2.683,9.400,9.400,0.1345


In [4]:
questions.head()

,question_id,concept,difficulty
0,Q001,INNER JOIN,0.589
1,Q002,NORMALIZATION,0.720
2,Q003,INNER JOIN,0.484
3,Q004,AGGREGATE FUNCTIONS,0.438
4,Q005,GROUP BY,0.393


In [5]:
questions.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   question_id  500 non-null    str    
 1   concept      500 non-null    str    
 2   difficulty   500 non-null    float64
dtypes: float64(1), str(2)
memory usage: 11.8 KB


In [6]:
questions.describe(include="all")

,question_id,concept,difficulty
count,500,500,500.000000
unique,500,20,NaN
top,Q001,HAVING,NaN
freq,1,33,NaN
mean,NaN,NaN,0.503694
std,NaN,NaN,0.158843
min,NaN,NaN,0.152000
25%,NaN,NaN,0.386500
50%,NaN,NaN,0.504000
75%,NaN,NaN,0.633000


In [7]:
questions["concept"].value_counts()

concept
HAVING                 33
SELECT                 32
WHERE                  31
VIEWS                  30
NORMALIZATION          29
ORDER BY               28
JOINS                  27
LEFT JOIN              27
AGGREGATE FUNCTIONS    25
TRIGGERS               25
GROUP BY               24
RIGHT JOIN             24
INNER JOIN             23
INDEXING               23
FOREIGN KEY            23
PRIMARY KEY            22
SUBQUERIES             21
STORED PROCEDURES      20
TRANSACTIONS           17
UNION                  16
Name: count, dtype: int64

In [8]:
print(
    questions["difficulty"].value_counts()
    .sort_index()
)

difficulty
0.152    1
0.153    1
0.156    1
0.158    1
0.171    1
        ..
0.818    1
0.827    1
0.828    1
0.835    1
0.846    1
Name: count, Length: 346, dtype: int64


## 5.6 — Calculate student concept performance

In [9]:
interactions["timestamp"] = pd.to_datetime(
    interactions["timestamp"]
)

interactions = interactions.sort_values(
    by=["student_id", "timestamp"]
).reset_index(drop=True)

In [11]:
concept_performance = (
    interactions
    .groupby(["student_id", "concept"])
    .agg(
        attempts=("correct", "count"),
        correct_answers=("correct", "sum"),
        accuracy=("correct", "mean"),
        average_gap=("days_since_practice", "mean")
    )
    .reset_index()
)

In [12]:
concept_performance.head(20)

,student_id,concept,attempts,correct_answers,accuracy,average_gap
0,S001,AGGREGATE FUNCTIONS,5,2,0.400000,27.441000
1,S001,FOREIGN KEY,4,1,0.250000,33.362750
2,S001,GROUP BY,5,1,0.200000,20.066600
3,S001,HAVING,3,0,0.000000,39.919000
4,S001,INDEXING,4,1,0.250000,35.362250
5,S001,INNER JOIN,6,1,0.166667,20.996333
6,S001,JOINS,4,0,0.000000,34.170500
7,S001,LEFT JOIN,3,0,0.000000,25.233000
8,S001,NORMALIZATION,4,2,0.500000,29.038000
9,S001,ORDER BY,4,4,1.000000,32.755250


## 5.7 — Identify weak concepts

In [13]:
weak_concepts = concept_performance[
    concept_performance["accuracy"] < 0.60
].copy()

print("Number of weak concept records:",
      len(weak_concepts))

weak_concepts.head(20)

Number of weak concept records: 1732


,student_id,concept,attempts,correct_answers,accuracy,average_gap
0,S001,AGGREGATE FUNCTIONS,5,2,0.400000,27.441000
1,S001,FOREIGN KEY,4,1,0.250000,33.362750
2,S001,GROUP BY,5,1,0.200000,20.066600
3,S001,HAVING,3,0,0.000000,39.919000
4,S001,INDEXING,4,1,0.250000,35.362250
5,S001,INNER JOIN,6,1,0.166667,20.996333
6,S001,JOINS,4,0,0.000000,34.170500
7,S001,LEFT JOIN,3,0,0.000000,25.233000
8,S001,NORMALIZATION,4,2,0.500000,29.038000
10,S001,PRIMARY KEY,7,4,0.571429,19.906714


## 5.8 — Load the forgetting model

In [14]:
import joblib

forgetting_model = joblib.load(
    "learntwin_forgetting_model.pkl"
)

print("Forgetting model loaded successfully!")

Forgetting model loaded successfully!


## 5.9 — Calculate forgetting risk for the student's concepts

In [15]:
concept_group = interactions.groupby(
    ["student_id", "concept"]
)

interactions["concept_prior_correct"] = (
    concept_group["correct"].cumsum()
    - interactions["correct"]
)

interactions["concept_prior_attempts"] = (
    concept_group.cumcount()
)

interactions["concept_prior_accuracy"] = np.where(
    interactions["concept_prior_attempts"] > 0,
    interactions["concept_prior_correct"] /
    interactions["concept_prior_attempts"],
    0.5
)

In [16]:
interactions["student_prior_correct"] = (
    interactions
    .groupby("student_id")["correct"]
    .cumsum()
    - interactions["correct"]
)

interactions["student_prior_attempts"] = (
    interactions.groupby("student_id").cumcount()
)

interactions["student_prior_accuracy"] = np.where(
    interactions["student_prior_attempts"] > 0,
    interactions["student_prior_correct"] /
    interactions["student_prior_attempts"],
    0.5
)

## 5.10 — Generate forgetting probabilities

In [17]:
forgetting_features = [
    "concept",
    "difficulty",
    "time_gap_days",
    "days_since_practice",
    "student_prior_attempts",
    "student_prior_accuracy",
    "concept_prior_attempts",
    "concept_prior_accuracy"
]

In [18]:
latest_concept_state = (
    interactions
    .sort_values("timestamp")
    .groupby(["student_id", "concept"])
    .tail(1)
    .copy()
)

In [19]:
latest_concept_state["forgetting_probability"] = (
    forgetting_model.predict_proba(
        latest_concept_state[forgetting_features]
    )[:, 1]
)

In [20]:
latest_concept_state[
    [
        "student_id",
        "concept",
        "correct",
        "concept_prior_accuracy",
        "days_since_practice",
        "forgetting_probability"
    ]
].head(20)

,student_id,concept,correct,concept_prior_accuracy,days_since_practice,forgetting_probability
7002,S071,SUBQUERIES,0,0.5,1.325,0.451983
6703,S068,TRANSACTIONS,0,0.5,4.061,0.380751
2104,S022,TRANSACTIONS,1,0.5,4.087,0.337902
8805,S089,UNION,0,0.5,10.022,0.312093
2607,S027,TRANSACTIONS,0,0.5,10.035,0.342842
511,S006,SELECT,0,0.5,11.937,0.343080
4611,S047,ORDER BY,1,0.5,12.938,0.370795
906,S010,UNION,0,0.0,10.360,0.364847
9514,S096,RIGHT JOIN,0,0.0,8.486,0.325394
7713,S078,UNION,1,0.5,20.135,0.366618


## 5.11 — Create a combined student-concept profile

In [22]:
student_concept_profile = concept_performance.merge(
    latest_concept_state[
        [
            "student_id",
            "concept",
            "days_since_practice",
            "forgetting_probability"
        ]
    ],
    on=["student_id", "concept"],
    how="left"
)

In [23]:
student_concept_profile.head(20)

,student_id,concept,attempts,correct_answers,accuracy,average_gap,days_since_practice,forgetting_probability
0,S001,AGGREGATE FUNCTIONS,5,2,0.400000,27.441000,7.027,0.336455
1,S001,FOREIGN KEY,4,1,0.250000,33.362750,10.409,0.438448
2,S001,GROUP BY,5,1,0.200000,20.066600,7.313,0.369125
3,S001,HAVING,3,0,0.000000,39.919000,29.845,0.603047
4,S001,INDEXING,4,1,0.250000,35.362250,3.540,0.466624
5,S001,INNER JOIN,6,1,0.166667,20.996333,6.815,0.469696
6,S001,JOINS,4,0,0.000000,34.170500,21.402,0.615172
7,S001,LEFT JOIN,3,0,0.000000,25.233000,38.887,0.603387
8,S001,NORMALIZATION,4,2,0.500000,29.038000,10.140,0.449495
9,S001,ORDER BY,4,4,1.000000,32.755250,37.258,0.636377


## Student
 ##  ↓
## Concept
 ##  ↓
## Accuracy
   ## +
## Time since practice
  ## +
## Forgetting probability

## 5.12 — Create a priority score

Weakness
+
Forgetting risk

In [25]:
student_concept_profile["weakness_score"] = (
    1 - student_concept_profile["accuracy"]
)

In [26]:
student_concept_profile["priority_score"] = (
    0.6 *
    student_concept_profile["weakness_score"]
    +
    0.4 *
    student_concept_profile["forgetting_probability"]
)

## 60% → current weakness
## 40% → forgetting risk

## 5.13 — See the highest-priority concepts

In [27]:
student_concept_profile = (
    student_concept_profile
    .sort_values(
        "priority_score",
        ascending=False
    )
)

student_concept_profile.head(20)

,student_id,concept,attempts,correct_answers,accuracy,average_gap,days_since_practice,forgetting_probability,weakness_score,priority_score
905,S046,SUBQUERIES,1,0,0.0,147.500000,147.500,0.993690,1.0,0.997476
1516,S077,TRANSACTIONS,1,0,0.0,146.438000,146.438,0.992034,1.0,0.996814
927,S047,UNION,1,0,0.0,156.194000,156.194,0.991567,1.0,0.996627
290,S015,RIGHT JOIN,1,0,0.0,156.214000,156.214,0.990711,1.0,0.996284
1004,S051,TRIGGERS,4,0,0.0,38.973250,138.198,0.990522,1.0,0.996209
1496,S076,TRANSACTIONS,1,0,0.0,140.344000,140.344,0.989815,1.0,0.995926
1303,S067,AGGREGATE FUNCTIONS,2,0,0.0,76.138500,144.740,0.988554,1.0,0.995422
1259,S064,SUBQUERIES,2,0,0.0,73.387000,122.134,0.988233,1.0,0.995293
1777,S090,VIEWS,2,0,0.0,86.201500,143.974,0.987771,1.0,0.995108
1794,S091,TRANSACTIONS,1,0,0.0,133.016000,133.016,0.987552,1.0,0.995021


## 5.14 — Select questions for the weak concept

In [28]:
student_id = "S001"

In [29]:
def recommend_questions(
    student_id,
    num_questions=5
):
    
    # Get this student's concept profile
    student_profile = student_concept_profile[
        student_concept_profile["student_id"] == student_id
    ].copy()
    
    if student_profile.empty:
        print("Student not found.")
        return pd.DataFrame()
    
    # Highest priority concept
    target_concept = (
        student_profile
        .sort_values(
            "priority_score",
            ascending=False
        )
        .iloc[0]["concept"]
    )
    
    # Get questions for that concept
    candidate_questions = questions[
        questions["concept"] == target_concept
    ].copy()
    
    # Remove questions already attempted
    attempted_questions = interactions[
        interactions["student_id"] == student_id
    ]["question_id"].unique()
    
    candidate_questions = candidate_questions[
        ~candidate_questions["question_id"]
        .isin(attempted_questions)
    ]
    
    # If no new questions remain
    if candidate_questions.empty:
        print(
            "No unattempted questions available "
            "for this concept."
        )
        return pd.DataFrame()
    
    # Return available questions
    return candidate_questions.head(
        num_questions
    )

## 5.15 — Test the recommender

In [30]:
print(
    interactions["student_id"]
    .unique()[:10]
)

<StringArray>
['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009',
 'S010']
Length: 10, dtype: str


In [31]:
test_student = interactions[
    "student_id"
].iloc[0]

print("Testing student:", test_student)

Testing student: S001


In [32]:
recommended = recommend_questions(
    test_student,
    num_questions=5
)

recommended

,question_id,concept,difficulty
66,Q067,TRANSACTIONS,0.606
93,Q094,TRANSACTIONS,0.694
94,Q095,TRANSACTIONS,0.782
101,Q102,TRANSACTIONS,0.659
110,Q111,TRANSACTIONS,0.674


## Student History
   ##   ↓
## Concept Accuracy
   ##     ↓
## Weakness Score
   ##   ↓
## Forgetting Prediction
   ##   ↓
## Forgetting Probability
   ##   ↓
## Priority Score
   ##   ↓
## Weakest / highest-risk Concept
   ##   ↓
## Find unattempted questions
   ##   ↓
## Recommend questions